# Museum Persona Lab
AA-01-03-E01 → AA-01-03-C01

Follow the book in order: construct and refine a role-based data-analysis request in 3.2, then use the separate evaluation notebooks in 3.3. We use the same eight synthetic visitor comments throughout. This notebook makes no live request unless you choose live mode.


**Downloaded package:** keep one private `.env` at the package root beside `COMPANION_VERSION`. The helper finds it automatically. When using this notebook alone, keep `.env` beside the notebook. The `get_completion()` interface is unchanged.

## OpenAI setup

Use the same setup as Section 2.3. The cell below creates a blank `openai.env.example` file without replacing an existing file. Open it in the notebook file browser, save a copy as `.env` at the package root, then manually replace both placeholders with your own key and model name. Do not put the key in a code cell or share the filled file.

A standard Python kernel needs `openai` and `python-dotenv` (`%pip install openai python-dotenv`). Set `MODE = "live"` to run your prompts through OpenAI. The Try It browser kernel uses supplied responses in `"offline"` mode; it cannot run the standard OpenAI SDK. A supplied response does not test an edited prompt.


## Start here

This notebook includes its own setup. You do not need to run another notebook first. Start at the top and run each code cell in order, including the setup cells. To run everything, choose **Run → Run All Cells**. After restarting the kernel, run the setup again before running the experiment.

The default mode uses supplied examples and needs no API key. For OpenAI requests, follow the configuration instructions in this notebook and use a standard Python kernel.


In [ ]:
# Shared offline/live request adapter. No API key is stored in this notebook.
# Code origin: get_completion(system_prompt, user_prompt) was introduced in Section 2.3.
# Notebook extension used from Section 3.2: supplied/live mode and interaction records.
# Evaluation notebooks reuse this helper without changing its two-argument interface.
import json
import os
import sys

MODE = "offline"
MODEL = ""
_client = None
request_records = []
OFFLINE_SAMPLES = {}

# Match supplied examples only when both messages are unchanged.
def _request_key(system_prompt, user_prompt):
    return json.dumps([system_prompt.strip(), user_prompt.strip()], ensure_ascii=False)

# Register a supplied teaching response; this does not contact OpenAI.
def register_sample(system_prompt, user_prompt, response):
    OFFLINE_SAMPLES[_request_key(system_prompt, user_prompt)] = response

# Locate one package-root configuration, while retaining standalone notebook use.
def _configuration_file():
    from pathlib import Path
    bases = [Path.cwd()]
    if "__file__" in globals():
        bases.insert(0, Path(__file__).resolve().parent)
    for base in bases:
        for folder in [base, *base.parents]:
            if (folder / "COMPANION_VERSION").is_file():
                return folder / ".env"
    # Standalone notebooks keep their configuration beside the notebook.
    return bases[0] / ".env"

# Load the private configuration and prepare an OpenAI client when live mode is chosen.
def configure_live(model=""):
    global MODE, MODEL, _client
    if sys.platform in {"emscripten", "wasi"}:
        raise RuntimeError("The OpenAI SDK needs a standard Python kernel. Keep offline mode in browser Python.")
    try:
        from openai import OpenAI
        from dotenv import load_dotenv
    except ImportError:
        raise RuntimeError("Install openai and python-dotenv in your Python kernel, then rerun setup.") from None
    # Read the key and model from the local file without printing the key.
    load_dotenv(_configuration_file())
    selected_model = model.strip() or os.getenv("OPENAI_MODEL", "").strip()
    if not selected_model or selected_model.startswith("YOUR_"):
        raise ValueError("Set OPENAI_MODEL in your .env file before choosing live mode.")
    key = os.getenv("OPENAI_API_KEY", "").strip()
    if not key or key.startswith("YOUR_"):
        raise ValueError("Copy your own OpenAI key into the .env file before choosing live mode.")
    _client = OpenAI(timeout=45.0, max_retries=0)
    MODEL = selected_model
    MODE = "live"
    print("OpenAI configured from .env. No key is printed.")


def get_completion(system_prompt, user_prompt):
    if MODE == "live":
        if _client is None:
            raise RuntimeError("Run configure_live(MODEL) in a private Python kernel first.")
        try:
            # Send role instructions separately from the user task and data.
            response = _client.responses.create(model=MODEL, instructions=system_prompt, input=user_prompt, max_output_tokens=1000, store=False)
        except Exception:
            raise RuntimeError("API request failed. Check your model, account access and connection; no key has been printed.") from None
        text = response.output_text
        print("[Live OpenAI response]")
    else:
        # Offline mode returns stored text, not a newly generated model answer.
        text = OFFLINE_SAMPLES.get(_request_key(system_prompt, user_prompt))
        if text is None:
            print("[No saved response for these edited instructions]")
            return "No model was called. Use OpenAI mode in a standard Python kernel, or label a response you write yourself."
        print("[Supplied teaching response — no API call]")
    # Keep messages and responses for evaluation; never record the API key.
    request_records.append({"mode": MODE, "model": MODEL if MODE == "live" else "supplied-example", "system_prompt": system_prompt, "user_prompt": user_prompt, "response": text})
    return text

# Release the API client after finishing the experiment.
def clear_live():
    global _client, MODE
    if _client is not None:
        _client.close()
    _client = None
    MODE = "offline"
    print("Live client cleared from this kernel.")

In [ ]:
from pathlib import Path

config_template = Path("openai.env.example")
if not config_template.exists():
    config_template.write_text("OPENAI_API_KEY=YOUR_OPENAI_API_KEY\nOPENAI_MODEL=YOUR_MODEL_NAME\n", encoding="utf-8")
print("Open openai.env.example, save a copy as .env, and replace the placeholders there.")


### A first OpenAI request

The following optional cell follows the book's request pattern. After preparing `.env` in a standard Python kernel, set `RUN_OPENAI_EXAMPLE = True` and run it. It makes one actual API request. Leave it False for supplied browser practice. Later exercises use `get_completion()` with separate system instructions and user requests.


In [ ]:
RUN_OPENAI_EXAMPLE = False
if RUN_OPENAI_EXAMPLE:
    import sys
    if sys.platform in {"emscripten", "wasi"}:
        raise RuntimeError("Run this OpenAI cell in a standard Python kernel; browser practice uses supplied responses.")
    import os
    from openai import OpenAI
    from dotenv import load_dotenv

    load_dotenv()
    client = OpenAI()
    response = client.responses.create(
        model=os.getenv("OPENAI_MODEL"),
        input="Explain what an AI agent is in a few sentences."
    )
    print(response.output_text)
    client.close()


In [ ]:
MODE = "offline"  # Set to "live" in a standard Python kernel after editing .env.
MODEL = ""        # Leave empty to use OPENAI_MODEL from .env.
if MODE == "live":
    configure_live(MODEL)
else:
    print("Supplied examples: no OpenAI request. Set MODE to live to use your .env configuration.")


## 3.2 — Crafting a Role-Based Prompt

A role gives the model a perspective. A task, context and output expectations help it use that perspective for a particular purpose. Include only instructions that help the task.

### The four elements

| Element | Question |
| --- | --- |
| **Role** | What perspective should the model adopt? |
| **Task** | What should it do? |
| **Context** | What background, audience and data matter? |
| **Output expectations** | What should the response contain and look like? |

These describe parts of the instructions, not four separate API message types.


### Example — Analyse Museum Visitor Feedback

Museum management wants to identify improvements from visitor feedback. Its summary should be grounded in the comments, rather than general advice about running a museum. Read the same eight synthetic comments as Listing 3.1 in the book. Each comment has an ID for later checking.


In [ ]:
feedback_text = """F01: The exhibition labels were too small for me to read comfortably.
F02: I wanted clearer explanations next to the objects.
F03: The labels used specialist terms without explaining them.
F04: I waited a long time at the ticket desk.
F05: The queue at the entrance moved slowly.
F06: I could not find a seat when I needed a rest.
F07: More benches along the exhibition route would help.
F08: The staff were welcoming and helped me find the gallery."""


### Identify the four prompt elements

The next cell matches Listing 3.2. Notice the audience and objective in `context`, and the required response in `output_expectations`. Keep the task and data unchanged when comparing stages.


In [ ]:
# Role: the perspective the model should adopt.
role = 'You are a museum visitor-feedback analyst.'

# Task: the work requested by the user.
task = 'Summarize this visitor feedback:'

# Context: the audience, objective and data boundary.
context = (
    "The audience is museum management deciding which visitor "
    "experiences to improve. Use only the supplied comments."
)

# Output expectations: the required response.
output_expectations = (
    "Summarize three recurring improvement themes in concise bullet points. "
    "Add one limitation."
)


### Assemble the messages

As in Chapter 2, `get_completion(system_prompt, user_prompt)` sends two messages. Put the role, context and output expectations in the system prompt. Put the task and feedback in the user request. The `f` strings insert the variable values. The helper supplies labelled examples in offline mode or calls your model in live mode.


In [ ]:
system_prompt = f"{role}\n{context}\n{output_expectations}"
user_prompt = f"{task}\n{feedback_text}"


### Predict, run and compare

Before running the comparison, predict what a generic response might omit. What difference do you expect from the analyst role? What should the management context and output expectations add?

The following optional setup cell registers the supplied responses for offline mode. It is not part of your prompt. Its stored text cannot show what a real model would do with an edited instruction.


In [ ]:
# Supplied teaching responses for the exact comparison prompts below.
# These are examples, not live model results.
register_sample('You are a helpful assistant.', 'Summarize this visitor feedback:\nF01: The exhibition labels were too small for me to read comfortably.\nF02: I wanted clearer explanations next to the objects.\nF03: The labels used specialist terms without explaining them.\nF04: I waited a long time at the ticket desk.\nF05: The queue at the entrance moved slowly.\nF06: I could not find a seat when I needed a rest.\nF07: More benches along the exhibition route would help.\nF08: The staff were welcoming and helped me find the gallery.', 'The comments mention exhibition labels, entrance waiting, seating and helpful staff.')
register_sample('You are a museum visitor-feedback analyst.', 'Summarize this visitor feedback:\nF01: The exhibition labels were too small for me to read comfortably.\nF02: I wanted clearer explanations next to the objects.\nF03: The labels used specialist terms without explaining them.\nF04: I waited a long time at the ticket desk.\nF05: The queue at the entrance moved slowly.\nF06: I could not find a seat when I needed a rest.\nF07: More benches along the exhibition route would help.\nF08: The staff were welcoming and helped me find the gallery.', 'Visitors report difficulties with labels, queues and seating. One comment praises the staff.')
register_sample('You are a museum visitor-feedback analyst.\nThe audience is museum management deciding which visitor experiences to improve. Use only the supplied comments.', 'Summarize this visitor feedback:\nF01: The exhibition labels were too small for me to read comfortably.\nF02: I wanted clearer explanations next to the objects.\nF03: The labels used specialist terms without explaining them.\nF04: I waited a long time at the ticket desk.\nF05: The queue at the entrance moved slowly.\nF06: I could not find a seat when I needed a rest.\nF07: More benches along the exhibition route would help.\nF08: The staff were welcoming and helped me find the gallery.', 'The recurring improvement areas are labels and explanations, entrance waiting and rest seating. These comments can guide management discussion but do not establish causes.')
register_sample('You are a museum visitor-feedback analyst.\nThe audience is museum management deciding which visitor experiences to improve. Use only the supplied comments.\nSummarize three recurring improvement themes in concise bullet points. Add one limitation.', 'Summarize this visitor feedback:\nF01: The exhibition labels were too small for me to read comfortably.\nF02: I wanted clearer explanations next to the objects.\nF03: The labels used specialist terms without explaining them.\nF04: I waited a long time at the ticket desk.\nF05: The queue at the entrance moved slowly.\nF06: I could not find a seat when I needed a rest.\nF07: More benches along the exhibition route would help.\nF08: The staff were welcoming and helped me find the gallery.', '- Labels and explanations: visitors wanted easier-to-read, clearer labels.\n- Entrance waiting: comments mention a slow queue and ticket desk.\n- Seating: comments ask for places to rest.\nLimitation: eight comments cannot establish the views of all visitors.')
register_sample('You are a museum visitor-feedback analyst.\nThe audience is museum management deciding which visitor experiences to improve. Use only the supplied comments.\nSummarize three recurring improvement themes in concise bullet points. Add one limitation.\nSupport each theme with the IDs of the relevant comments.', 'Summarize this visitor feedback:\nF01: The exhibition labels were too small for me to read comfortably.\nF02: I wanted clearer explanations next to the objects.\nF03: The labels used specialist terms without explaining them.\nF04: I waited a long time at the ticket desk.\nF05: The queue at the entrance moved slowly.\nF06: I could not find a seat when I needed a rest.\nF07: More benches along the exhibition route would help.\nF08: The staff were welcoming and helped me find the gallery.', '- Labels and explanations: F01, F02 and F03.\n- Entrance waiting: F04 and F05.\n- Rest seating: F06 and F07.\nF08 praises helpful staff.\nLimitation: these eight comments do not establish the views of all visitors.')
register_sample('You are a museum visitor-feedback analyst.\nThe audience is museum visitors seeking a brief overview. Use only the supplied comments.\nSummarize three recurring improvement themes in concise bullet points. Add one limitation.', 'Summarize this visitor feedback:\nF01: The exhibition labels were too small for me to read comfortably.\nF02: I wanted clearer explanations next to the objects.\nF03: The labels used specialist terms without explaining them.\nF04: I waited a long time at the ticket desk.\nF05: The queue at the entrance moved slowly.\nF06: I could not find a seat when I needed a rest.\nF07: More benches along the exhibition route would help.\nF08: The staff were welcoming and helped me find the gallery.', 'Visitors mentioned exhibition labels, entrance waiting and rest seating; one praised helpful staff. This is a small collection of comments, not a promise that the museum has changed its services.')


**Baseline:** keep the same task and feedback, using a generic assistant. Read the response and note what it does or leaves out.


In [ ]:
baseline_system_prompt = "You are a helpful assistant."
baseline_response = get_completion(baseline_system_prompt, user_prompt)
print(baseline_response)


**Add the role:** change only the system instruction to the museum analyst role. Predict a difference, then compare with the baseline.


In [ ]:
role_system_prompt = role
role_response = get_completion(role_system_prompt, user_prompt)
print(role_response)


**Add context:** tell the analyst who will use the summary and why. Keep the user request unchanged. Predict what this should change before running.


In [ ]:
context_system_prompt = f"{role}\n{context}"
context_response = get_completion(context_system_prompt, user_prompt)
print(context_response)


**Add output expectations:** now use the complete system prompt assembled above. Predict what should appear in the response, then run the request from Listing 3.3.


In [ ]:
constructed_system_prompt = system_prompt
constructed_response = get_completion(system_prompt, user_prompt)
print(constructed_response)


### Explain what changed

Identify `role`, `task`, `context` and `output_expectations` in the two messages. Check that the actual feedback is included. Explain one observed difference between stages, using the comments to check the claims. A more professional tone does not necessarily mean greater accuracy.

Keep `constructed_system_prompt` and its response for the refinement below. In offline mode, a changed prompt without a saved response is reported explicitly; it does not produce a new model answer.


### Refine the same prompt
Choose instructions that serve the task, audience and intended outcome. Read the response you already obtained. Can management trace each theme back to a comment? Predict the effect of one added evidence instruction. This is the final checkpoint of 3.2; do not restart with a new vague request.

In [ ]:
refined_system_prompt = constructed_system_prompt + "\nSupport each theme with the IDs of the relevant comments."
refined_response = get_completion(refined_system_prompt, user_prompt)
print(refined_response)

Compare the two responses against the comments. Does the added instruction make the claims easier to check? Explain the specific weakness addressed. Optional: change only the audience clause to museum visitors, predict the effect and run your revision in live mode; label offline results honestly.

Evaluation continues in Section 3.3 using separate, self-contained experiment notebooks. Save your construction/refinement observations; the new notebooks include all their own inputs and references.